IMPORTS

In [6]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from psycopg2.extras import execute_batch
import pandas as pd

FILES LOADING

In [7]:
holidays = pd.read_csv("holidays_events.csv")
oil = pd.read_csv("oil.csv")
stores = pd.read_csv("stores.csv")
transactions = pd.read_csv("transactions.csv")
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submission = pd.read_csv("sample_submission.csv")

PASSWORD

In [8]:
with open('pwd.txt', 'w') as file:
    file.write(input('Enter the password: '))

with open('pwd.txt', 'r') as file:
    pwd = file.read()

TASK 1

1A - the exercise asked for: "frequency, percentage of data that is missing (total and per column)". Oil is the most interesting table to check because it actually has missing values: 43 NULL prices out of 1,218 rows

1B - also frequency, percentage of data that is missing (total and per column)". Transactions is my choice and very important table to observe. 

1C - also frequency, percentage of data that is missing (total and per column)". Chose Holidays.

1D - "gaps in oil price time series". In the task it was said to use LEFT JOIN, that is why i came up with idea to detect missing values.

1E - so we needed to identify missing records, but first of all we need to use CROSS JOIN and create 91,152 rows, which represent every store-date pair that should exist. Without this, you cannot understand what is missing. 

1F - we also needed to do something with GROUP BY, so i decided to use the function to group by date all holiday rows for the same day together, it can show us where the types actually conflict

In [ ]:
# Connection details
DB_NAME = "postgres"  
DB_USER = "postgres"
DB_PASS = pwd
DB_HOST = "localhost"
DB_PORT = "5432"


# ══════════════════════════════════════════════════════════════════════════════
# PART 1 — SQL DATA QUALITY REPORT
# ══════════════════════════════════════════════════════════════════════════════

print("\n\n" + "="*80)
print("  TASK 1")
print("="*80)

# 1A. NULL Counts — Oil
# We need to count how many oil price values are missing (NULL) and what percentage
print('_________________________________________________THE START OF 1A___________________________________________________________________________')

def null_counts_oil(oil: pd.DataFrame):

    conn = psycopg2.connect(dbname = DB_NAME,
                            user = DB_USER,
                            password = DB_PASS,
                            host = DB_HOST,
                            port = DB_PORT)

    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)

    cur = conn.cursor()

    # Drop the table if it exists from a previous run, then create it fresh
    cur.execute("DROP TABLE IF EXISTS oil;")
    cur.execute("""
        CREATE TABLE oil (
            date TEXT,
            dcoilwtico DOUBLE PRECISION
        );
    """)

    # Insert data row by row from the pandas DataFrame
    # pd.isna() checks for NaN — we convert NaN to None
    for _, row in oil.iterrows():
        val = None if pd.isna(row['dcoilwtico']) else row['dcoilwtico']
        cur.execute("INSERT INTO oil VALUES (%s, %s)", (row['date'], val))
    conn.commit()

    # COUNT(*) counts all rows including NULLs
    # COUNT(dcoilwtico) counts only rows where price is NOT NULL
    # Subtracting them, so we get the actual number
    # 100.0, because we need decimal division, not integer division
    cur.execute("""
        SELECT
            COUNT(*)                                        AS total_rows,
            COUNT(*) - COUNT(dcoilwtico)                    AS null_oil_price,
            ROUND(100.0 * (COUNT(*) - COUNT(dcoilwtico))
                         / COUNT(*), 2)                     AS pct_null_oil
        FROM oil;
    """)

    # cur.fetchall() returns all result rows as a list of tuples
    data = cur.fetchall()

    # cur.description holds ionformation about result columns
    columns = [desc[0] for desc in cur.description]

    # Combine column names + data into a pandas DataFrame
    result_df = pd.DataFrame(data, columns=columns)

    cur.execute("DROP TABLE oil;")
    conn.commit()
    cur.close()
    conn.close()
    return result_df

print(null_counts_oil(oil))
print('_________________________________________________THE END OF 1A___________________________________________________________________________')


# 1B. NULL Counts Transactions
# We need to verify that the transactions table has no NULLs in any column

print('_________________________________________________THE START OF 1B___________________________________________________________________________')

def null_counts_transactions(transactions: pd.DataFrame):

    conn = psycopg2.connect(dbname = DB_NAME,
                            user = DB_USER,
                            password = DB_PASS,
                            host = DB_HOST,
                            port = DB_PORT)

    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    cur = conn.cursor()

    cur.execute("DROP TABLE IF EXISTS transactions;")
    cur.execute("""
        CREATE TABLE transactions (
            date TEXT,
            store_nbr INT,
            transactions INT
        );
    """)

    for _, row in transactions.iterrows():
        cur.execute(
            "INSERT INTO transactions VALUES (%s, %s, %s)",
            (row['date'], row['store_nbr'], row['transactions'])
        )
    conn.commit()

    # Same COUNT(*) - COUNT(column) pattern on every column
    # All three should come back 0
    cur.execute("""
        SELECT
            COUNT(*)                            AS total_rows,
            COUNT(*) - COUNT(date)              AS null_date,
            COUNT(*) - COUNT(store_nbr)         AS null_store_nbr,
            COUNT(*) - COUNT(transactions)      AS null_transactions
        FROM transactions;
    """)
    data = cur.fetchall()
    columns = [desc[0] for desc in cur.description]
    result_df = pd.DataFrame(data, columns=columns)

    cur.execute("DROP TABLE transactions;")
    conn.commit()
    cur.close()
    conn.close()
    return result_df

print(null_counts_transactions(transactions))
print('_________________________________________________THE END OF 1B___________________________________________________________________________')


# 1C. NULL Counts Holidays
# We need to check all six columns of the holidays table for NULLs
print('_________________________________________________THE START OF 1C___________________________________________________________________________')

def null_counts_holidays(holidays: pd.DataFrame):

    conn = psycopg2.connect(dbname = DB_NAME,
                            user = DB_USER,
                            password = DB_PASS,
                            host = DB_HOST,
                            port = DB_PORT)

    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    cur = conn.cursor()

    cur.execute("DROP TABLE IF EXISTS holidays_events;")
    cur.execute("""
        CREATE TABLE holidays_events (
            date TEXT, type TEXT, locale TEXT,
            locale_name TEXT, description TEXT, transferred BOOLEAN
        );
    """)

    # bool(row['transferred']) converts pandas boolean to Python bool
    # because the column is defined as boolean in PostgreSQL
    for _, row in holidays.iterrows():
        cur.execute(
            "INSERT INTO holidays_events VALUES (%s, %s, %s, %s, %s, %s)",
            (row['date'], row['type'], row['locale'],
             row['locale_name'], row['description'], bool(row['transferred']))
        )
    conn.commit()

    # Check all six columns at once — all should come back 0
    cur.execute("""
        SELECT
            COUNT(*)                            AS total_rows,
            COUNT(*) - COUNT(date)              AS null_date,
            COUNT(*) - COUNT(type)              AS null_type,
            COUNT(*) - COUNT(locale)            AS null_locale,
            COUNT(*) - COUNT(locale_name)       AS null_locale_name,
            COUNT(*) - COUNT(description)       AS null_description,
            COUNT(*) - COUNT(transferred)       AS null_transferred
        FROM holidays_events;
    """)
    data = cur.fetchall()
    columns = [desc[0] for desc in cur.description]
    result_df = pd.DataFrame(data, columns=columns)

    cur.execute("DROP TABLE holidays_events;")
    conn.commit()
    cur.close()
    conn.close()
    return result_df

print(null_counts_holidays(holidays))
print('_________________________________________________THE END OF 1C___________________________________________________________________________')


# 1D. Oil Missing Calendar Dates
# Goal: find dates that are COMPLETELY ABSENT from the oil table
# (different from 1A which found rows that exist but have NULL prices)
print('_________________________________________________THE START OF 1D___________________________________________________________________________')

def oil_missing_dates(oil: pd.DataFrame):


    conn = psycopg2.connect(dbname = DB_NAME,
                            user = DB_USER,
                            password = DB_PASS,
                            host = DB_HOST,
                            port = DB_PORT)

    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    cur = conn.cursor()

    cur.execute("DROP TABLE IF EXISTS oil;")
    cur.execute("CREATE TABLE oil (date TEXT, dcoilwtico DOUBLE PRECISION);")
    for _, row in oil.iterrows():
        val = None if pd.isna(row['dcoilwtico']) else row['dcoilwtico']
        cur.execute("INSERT INTO oil VALUES (%s, %s)", (row['date'], val))
    conn.commit()

    # generate_series() creates every date from min to max — our "complete calendar"
    # ::DATE casts text to a real date type (generate_series needs actual dates)
    # LEFT JOIN keeps every calendar date — if oil has no match, columns become NULL
    # TO_CHAR() formats dates as 'YYYY-MM' for grouping by month
    # COUNT(o.date) skips NULLs = counts only dates that have oil data
    cur.execute("""
        SELECT
            TO_CHAR(c.cal_date, 'YYYY-MM')                AS year_month,
            COUNT(*)                                      AS calendar_days,
            COUNT(o.date)                                 AS oil_rows_present,
            COUNT(*) - COUNT(o.date)                      AS missing_days,
            ROUND(100.0 * (COUNT(*) - COUNT(o.date))
                         / COUNT(*), 1)                   AS pct_missing
        FROM generate_series(
                (SELECT MIN(date)::DATE FROM oil),
                (SELECT MAX(date)::DATE FROM oil),
                '1 day'::INTERVAL
             ) AS c(cal_date)
        LEFT JOIN oil o ON o.date::DATE = c.cal_date
        GROUP BY TO_CHAR(c.cal_date, 'YYYY-MM')
        ORDER BY year_month;
    """)
    data = cur.fetchall()
    columns = [desc[0] for desc in cur.description]
    result_df = pd.DataFrame(data, columns=columns)

    cur.execute("DROP TABLE oil;")
    conn.commit()
    cur.close()
    conn.close()
    return result_df

print(oil_missing_dates(oil))
print('_________________________________________________THE END OF 1D___________________________________________________________________________')


# 1E. Transactions. Missing Dates Per Store
# We need to find for each store how many days are missing from the transactions table
# A store can have no row for a given day, that s not a NULL, it is an absent record
print('_________________________________________________THE START OF 1E__________________________________________________________________________')

def missing_dates_per_store(transactions: pd.DataFrame, stores: pd.DataFrame):

    conn = psycopg2.connect(dbname = DB_NAME,
                            user = DB_USER,
                            password = DB_PASS,
                            host = DB_HOST,
                            port = DB_PORT)

    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    cur = conn.cursor()

    # This function needs two tables: transactions and stores
    cur.execute("DROP TABLE IF EXISTS transactions;")
    cur.execute("CREATE TABLE transactions (date TEXT, store_nbr INT, transactions INT);")
    for _, row in transactions.iterrows():
        cur.execute(
            "INSERT INTO transactions VALUES (%s, %s, %s)",
            (row['date'], row['store_nbr'], row['transactions'])
        )

    cur.execute("DROP TABLE IF EXISTS stores;")
    cur.execute("CREATE TABLE stores (store_nbr INT PRIMARY KEY, city TEXT, state TEXT, type TEXT, cluster INT);")
    for _, row in stores.iterrows():
        cur.execute(
            "INSERT INTO stores VALUES (%s, %s, %s, %s, %s)",
            (row['store_nbr'], row['city'], row['state'], row['type'], row['cluster'])
        )
    conn.commit()

    # CROSS JOIN stores × calendar = every possible store-date combination (91,152 rows)
    # LEFT JOIN transactions attaches actual data and NULL where a store-date pair is missing
    # Two join conditions: must match on both store_nbr and date
    # GROUP BY store gives one summary row per store
    # ORDER BY missing_days DESC puts worst stores first
    cur.execute("""
        SELECT
            s.store_nbr,
            COUNT(*)                          AS expected_days,
            COUNT(t.transactions)             AS actual_days,
            COUNT(*) - COUNT(t.transactions)  AS missing_days,
            ROUND(100.0 * (COUNT(*) - COUNT(t.transactions))
                         / COUNT(*), 1)       AS pct_missing
        FROM stores s
        CROSS JOIN generate_series(
                (SELECT MIN(date)::DATE FROM transactions),
                (SELECT MAX(date)::DATE FROM transactions),
                '1 day'::INTERVAL
             ) AS c(cal_date)
        LEFT JOIN transactions t
               ON t.store_nbr = s.store_nbr
              AND t.date::DATE = c.cal_date
        GROUP BY s.store_nbr
        ORDER BY missing_days DESC
        LIMIT 20;
    """)
    data = cur.fetchall()
    columns = [desc[0] for desc in cur.description]
    result_df = pd.DataFrame(data, columns = columns)

    cur.execute("DROP TABLE transactions;")
    cur.execute("DROP TABLE stores;")
    conn.commit()
    cur.close()
    conn.close()
    return result_df

print(missing_dates_per_store(transactions, stores))
print('_________________________________________________THE END OF 1E__________________________________________________________________________')


# 1F. Holidays. Same Date, Multiple Types
# We need to find dates where more than one holiday type was recorded
# For example, Dec 22 is both "Holiday" (local) and "Additional" (national)
print('_________________________________________________THE START OF 1F__________________________________________________________________________')

def holiday_conflicting_types(holidays: pd.DataFrame):

    conn = psycopg2.connect(dbname = DB_NAME,
                            user = DB_USER,
                            password = DB_PASS,
                            host = DB_HOST,
                            port = DB_PORT)

    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    cur = conn.cursor()

    cur.execute("DROP TABLE IF EXISTS holidays_events;")
    cur.execute("""
        CREATE TABLE holidays_events (
            date TEXT, type TEXT, locale TEXT,
            locale_name TEXT, description TEXT, transferred BOOLEAN
        );
    """)
    for _, row in holidays.iterrows():
        cur.execute(
            "INSERT INTO holidays_events VALUES (%s, %s, %s, %s, %s, %s)",
            (row['date'], row['type'], row['locale'],
             row['locale_name'], row['description'], bool(row['transferred']))
        )
    conn.commit()

    # GROUP BY date — one group per date
    # HAVING COUNT(DISTINCT type) > 1 — only keep dates with more than one type
    # DISTINCT is important: without it COUNT(type) would count duplicates too
    # STRING_AGG() joins values within a group like Python's ', '.join()
    cur.execute("""
        SELECT
            date,
            COUNT(*)                                AS n_rows,
            STRING_AGG(DISTINCT type, ', ')         AS types_seen,
            STRING_AGG(DISTINCT locale, ', ')       AS locales_seen,
            STRING_AGG(description, ' | ')          AS descriptions
        FROM holidays_events
        GROUP BY date
        HAVING COUNT(DISTINCT type) > 1
        ORDER BY date;
    """)
    data = cur.fetchall()
    columns = [desc[0] for desc in cur.description]
    result_df = pd.DataFrame(data, columns=columns)

    cur.execute("DROP TABLE holidays_events;")
    conn.commit()
    cur.close()
    conn.close()
    return result_df

print(holiday_conflicting_types(holidays))
print('_________________________________________________THE END OF 1F__________________________________________________________________________')



  TASK 1
_________________________________________________THE START OF 1A___________________________________________________________________________
   total_rows  null_oil_price pct_null_oil
0        1218              43         3.53
_________________________________________________THE END OF 1A___________________________________________________________________________
_________________________________________________THE START OF 1B___________________________________________________________________________
   total_rows  null_date  null_store_nbr  null_transactions
0       83488          0               0                  0
_________________________________________________THE END OF 1B___________________________________________________________________________
_________________________________________________THE START OF 1C___________________________________________________________________________
   total_rows  null_date  null_type  null_locale  null_locale_name  \
0         350    

TASK 2

The exercise asks to add day of week, is_weekend, month, quarter, year, week of year, and days to nearest holiday. These are all features that capture time-based patterns in sales data — weekly cycles (people shop more on weekends), seasonal patterns (December vs February), and holiday effects (sales spike before holidays, dip on the day itself).

What it does: Converts date strings to datetime, extracts each calendar attribute using pandas' .dt accessor, computes the distance to the nearest National holiday by subtracting each transaction date from every holiday date and picking the minimum absolute difference. Saves the result as a new CSV with 8 additional columns.

In [10]:
# we work on a copy so the original transactions DataFrame stays untouched
df = transactions.copy()


# pd.to_datetime() converts data into real datetime objects
df['date'] = pd.to_datetime(df['date'])

# basic calendar columns 

# dayofweek can be described like 0=Monday, 1=Tuesday, ... 6=Sunday
df['day_of_week'] = df['date'].dt.dayofweek

# regular name like "Monday", "Tuesday" useful for reports
df['day_name'] = df['date'].dt.day_name()

# 1 if Saturday or Sunday, 0 otherwise
# .isin([5,6]) returns True/False, .astype(int) converts to 1 or 0
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

# Month number 1-12
df['month'] = df['date'].dt.month

# Quarter 1-4
df['quarter'] = df['date'].dt.quarter

# 4-digit year
df['year'] = df['date'].dt.year

# week number 1-53. Weeks start on Monday
# .astype(int) is needed because isocalendar() returns a smth strange, i do not like it tbh
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)

# days to nearest holiday

# filter to national holidays only
# Local/Regional holidays only affect specific cities so we skip them here.
national_holidays = holidays[holidays['locale'] == 'National']['date'].values

# some dates appear more than once in the table, so we deduplicate with .unique(),
# then we convert to datetime, and sort
national_holidays = pd.to_datetime(pd.Series(national_holidays).unique()).sort_values()

def days_to_the_nearest_holiday(date_series, holiday_dates):

    # convert holiday dates to numpy datetime64 array for convenience
    hol_arr = holiday_dates.values.astype('datetime64[D]')
    result  = []

    for element in date_series:
        # convert the current date to the same numpy type
        date = element.to_datetime64().astype('datetime64[D]')

        # subtract current date from all holidays at once
        # abs() makes negative values positive
        # .astype(int) converts from this delta to teh int
        diffs = abs((hol_arr - date).astype(int))

        # .min() picks the smallest distance because we need nearest holiday
        # so if we got 0 = today is a holiday, 1 = one day away
        result.append(int(diffs.min()))

    return result

print("Computing days_to_nearest_holiday...")
df['days_to_nearest_holiday'] = days_to_the_nearest_holiday(df['date'], national_holidays)

print(df.head(10))

# save the enriched DataFrame to CSV, we need that
# index = False prevents pandas from writing its internal row numbers as a column
df.to_csv("transactions_enriched.csv", index = False)
print("\n✓ Saved to transactions_enriched.csv")


Computing days_to_nearest_holiday...
        date  store_nbr  transactions  day_of_week   day_name  is_weekend  \
0 2013-01-01         25           770            1    Tuesday           0   
1 2013-01-02          1          2111            2  Wednesday           0   
2 2013-01-02          2          2358            2  Wednesday           0   
3 2013-01-02          3          3487            2  Wednesday           0   
4 2013-01-02          4          1922            2  Wednesday           0   
5 2013-01-02          5          1903            2  Wednesday           0   
6 2013-01-02          6          2143            2  Wednesday           0   
7 2013-01-02          7          1874            2  Wednesday           0   
8 2013-01-02          8          3250            2  Wednesday           0   
9 2013-01-02          9          2940            2  Wednesday           0   

   month  quarter  year  week_of_year  days_to_nearest_holiday  
0      1        1  2013             1             

TASK 3 & 4

Значит, для первой задачи основная проблема и основной тейк был в том, чтобы как-то составить список вообще всех дат, которые не включены в CSV-файл. 

Для этого, соответственно, я использовал создание новой таблицы, потому что это оптимальный способ, он тратит меньше всего памяти и в целом быстрее всего работает. 

После этого я просто скомпилировал две таблицы, опять же, это самый удобный, самый надежный способ. 

касательно интерполяции линейной и backfill-forwardfill, я опять же, всегда сначала открываю документацию, смотрю, есть ли такое в библиотеке. Тут мне повезло, это в библиотеке было, поэтому я использовал встроенные функции для того, чтобы упростить работу себе и снизить нагрузку на систему на память в течение работы кода. 

В второй части, естественно, основной проблемой было, как мне посчитать разницу между днями, поэтому я перевел ячейки с датами в значение дат и составил общий список дат с шагом в день для того, чтобы смотреть, сколько между ними пустых ячеек. Через эти пустые ячейки я смог посчитать и подставить значение в соответствующие столбцы, сколько дней с предыдущего праздника прошло и сколько дней до следующего. 

Касательно того, что я перевел TrueFalse в значение 0/1, это необязательный шаг, и в коде он не имеет реального применения, однако это все равно полезно, если, например, с этим кодом в дальнейшем требуется, если с этой таблицей в дальнейшем требуется делать какие-либо манипуляции, потому что это имеет много плюсов.

In [11]:
df = pd.read_csv("oil.csv", parse_dates=["date"])
df = df.set_index("date")
#читаем файлик и переводим даты в формат дат, переводим в индексы столбец дат
full_range = pd.date_range(df.index.min(), df.index.max(), freq="D")
df = df.reindex(full_range)
#делаем список вообще всех дат с шагом в день и сопоставляем с цсв файлом чтобы добавить неуказанные дни
df["price_interp"] = df["dcoilwtico"].interpolate(method="linear")
df["price_interp"] = df["price_interp"].ffill().bfill()
df = df.reset_index().rename(columns={"index": "date"})
#сначала линейная интерполяция (документация в помощь), потом bforward back fill чтобы NaN убрать, потом убираем индекс он больше не нужен и переназываем колонкц индекс в date
df.to_csv("oil.csv", index=False)
#импортируем новый файл в другую директорию




df = pd.read_csv("holidays_events.csv", parse_dates=["date"])
#читаем файлик и переводим даты в формат дат, переводим в индексы столбец дат

df["transfer_day"] = df["transferred"].astype(int)
#делаем из тру фолз цифры (тру 1 фолз 0)
holidays = df[df["type"].isin(["Holiday", "Additional"])].copy()
holidays = holidays.sort_values("date")
#создаем новую таблицу где только холидей и аддишнл дни и сортируем по датам
holidays["days_before_holiday"] = (holidays["date"].shift(-1) - holidays["date"]).dt.days
#берем некст дату потом вычитаем предыдущую и разницу переводим в дни чтобы понять сколько дней до следующего праздника
holidays["days_after_holiday"] = (holidays["date"] - holidays["date"].shift(1)).dt.days
#тож самое только для предыдущего праздника
df = df.merge(holidays[["date","days_before_holiday","days_after_holiday"]], on="date", how="left")
#лефт join merge
df.to_csv("holidays_events.csv", index=False)
#переносим в новый файл в директории

TASK 5 & 6 & 7


1. We read data through pandas, because it's easier than using SQL

2. Delete the tables so that there are no errors if the function is run more than once

3. Create tables via "IF NOT EXISTS" so as not to get an error when starting the program

4. When filling out the table, we take certain columns so that there are no errors when updating the input data

5. We fill in the table using “copy_expert()”, since the usual “executemany” does this for a long time for a table with about 3 million rows (“train”) – filling in the table was one of the main problems

6. When solving tasks, we select specific columns, and not all of them, again, so that changing the input data does not affect the decision or give an error (deleting part of the data)

7. **TASK 5**: JOIN two tables and use the window functions to find the parameters we need. We use window functions to make it easier to compare values and, if necessary, combine them with other tables. We use "PARTITION BY" to group by specific groups of data. To extract the year, we use "EXTRACT" with "::INT" since "EXTRACT" returns a real value, which cannot be the year

8. **TASK 6**: Again, we use window functions to preserve the original appearance of the table. Using "PARTITION BY", we divide the data into certain "store-family" ("store-item") groups in order to count the parameters without mixing different data. "BETWEEN" is used to create a specific time frame. We use "STDDEV_SAMP" because we have data for a specific sample and not for the entire "population". For the coefficient of variation, we use the "CASE" expression/method to eliminate division by zero errors

9. **TASK 7**: The percentage of change is also calculated using the "CASE" expression/method to remove division by zero errors and incorrect values in case of insufficient accumulation of days for counting. In the end, we paint separate windows so that the code looks less bulky and is more readable

10. We calculate the median separately in pandas, since “PERCENTILE_CONT” cannot be used as a window function with “OVER” (it returned the error: “FeatureNotSupported: ERROR: the percentle_cont sorting aggregate does not support OVER LINE 15: PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY sales) OVER ...”). When grouping, we use “group_keys=False” and when calculating “reset_index()” to make it easier to combine the result with a common data table

11. At the end, we create a table with the results using pandas

In [ ]:
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password=pwd,
    host="localhost",
    port="5432",
)
conn.autocommit = True
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS stores;")
cur.execute("DROP TABLE IF EXISTS transactions;")
cur.execute("DROP TABLE IF EXISTS train;")

cur.execute("""
CREATE TABLE stores (
    store_nbr INTEGER PRIMARY KEY,
    city      VARCHAR(200),
    state     VARCHAR(200),
    type      VARCHAR(200),
    cluster   INTEGER
);
""")

with open("stores.csv", "r") as f:
    cur.copy_expert(
        """
        COPY stores (store_nbr, city, state, type, cluster)
        FROM STDIN WITH (FORMAT csv, HEADER true)
        """,
        f,
    )

cur.execute("""
CREATE TABLE transactions (
    date         DATE,
    store_nbr    INTEGER,
    transactions INTEGER
);
""")

with open("transactions.csv", "r") as f:
    cur.copy_expert(
        """
        COPY transactions (date, store_nbr, transactions)
        FROM STDIN WITH (FORMAT csv, HEADER true)
        """,
        f,
    )

cur.execute("""
CREATE TABLE train (
    id INTEGER PRIMARY KEY,
    date DATE,
    store_nbr INTEGER,
    family VARCHAR(200),
    sales REAL,
    onpromotion INTEGER
);
""")

with open("train.csv", "r") as f:
    cur.copy_expert(
        """
        COPY train (id, date, store_nbr, family, sales, onpromotion)
        FROM STDIN WITH (FORMAT csv, HEADER true)
        """,
        f,
    )

# ================== TASK 5 ==================

cur.execute("""
SELECT
    tr.date,
    tr.store_nbr,
    tr.transactions,
    s.city,
    s.state,
    s.type,
    s.cluster,
    
    AVG(tr.transactions) OVER (PARTITION BY s.type) AS avg_daily_sales_by_store_type,
    EXTRACT(YEAR FROM MIN(tr.date) OVER (PARTITION BY tr.store_nbr))::INT AS store_opening_year,
    AVG(tr.transactions) OVER (PARTITION BY s.cluster) AS avg_daily_transactions_by_cluster
    
FROM transactions tr
JOIN stores s
  ON tr.store_nbr = s.store_nbr;
""")

rows_5 = cur.fetchall()

cols_5 = [
    "date",
    "store_nbr",
    "transactions",
    "city",
    "state",
    "type",
    "cluster",
    "avg_daily_sales_by_store_type",
    "store_opening_year",
    "avg_daily_transactions_by_cluster",
]

df_5 = pd.DataFrame(rows_5, columns=cols_5)
df_5.to_csv("stores-transactions_features.csv", index=False)

# ================== TASK 6 и 7 ==================

cur.execute("""
SELECT
    date,
    store_nbr,
    family,
    sales,
    onpromotion,

    AVG(sales) OVER d_7  AS mean_7_days,
    AVG(sales) OVER d_14 AS mean_14_days,
    AVG(sales) OVER d_30 AS mean_30_days,
    AVG(sales) OVER d_60 AS mean_60_days,
    AVG(sales) OVER d_90 AS mean_90_days,

    -- медианы убрали из SQL, посчитаем их в pandas

    STDDEV_SAMP(sales) OVER d_7  AS std_7_days,
    STDDEV_SAMP(sales) OVER d_14 AS std_14_days,
    STDDEV_SAMP(sales) OVER d_30 AS std_30_days,
    STDDEV_SAMP(sales) OVER d_60 AS std_60_days,
    STDDEV_SAMP(sales) OVER d_90 AS std_90_days,

    MIN(sales) OVER d_7  AS min_7_days,
    MAX(sales) OVER d_7  AS max_7_days,

    MIN(sales) OVER d_14 AS min_14_days,
    MAX(sales) OVER d_14 AS max_14_days,

    MIN(sales) OVER d_30 AS min_30_days,
    MAX(sales) OVER d_30 AS max_30_days,

    MIN(sales) OVER d_60 AS min_60_days,
    MAX(sales) OVER d_60 AS max_60_days,

    MIN(sales) OVER d_90 AS min_90_days,
    MAX(sales) OVER d_90 AS max_90_days,

    CASE
        WHEN AVG(sales) OVER d_7 = 0
        THEN NULL
        ELSE STDDEV_SAMP(sales) OVER d_7 / AVG(sales) OVER d_7
    END AS coefficient_var_7_days,

    CASE
        WHEN AVG(sales) OVER d_14 = 0
        THEN NULL
        ELSE STDDEV_SAMP(sales) OVER d_14 / AVG(sales) OVER d_14
    END AS coefficient_var_14_days,

    CASE
        WHEN AVG(sales) OVER d_30 = 0
        THEN NULL
        ELSE STDDEV_SAMP(sales) OVER d_30 / AVG(sales) OVER d_30
    END AS coefficient_var_30_days,

    CASE
        WHEN AVG(sales) OVER d_60 = 0
        THEN NULL
        ELSE STDDEV_SAMP(sales) OVER d_60 / AVG(sales) OVER d_60
    END AS coefficient_var_60_days,

    CASE
        WHEN AVG(sales) OVER d_90 = 0
        THEN NULL
        ELSE STDDEV_SAMP(sales) OVER d_90 / AVG(sales) OVER d_90
    END AS coefficient_var_90_days,

    LAG(sales, 1)  OVER p AS lag_1_day,
    LAG(sales, 2)  OVER p AS lag_2_days,
    LAG(sales, 3)  OVER p AS lag_3_days,
    LAG(sales, 7)  OVER p AS lag_7_days,
    LAG(sales, 14) OVER p AS lag_14_days,
    LAG(sales, 28) OVER p AS lag_28_days,
    LAG(sales, 365)OVER p AS lag_365_days,

    CASE
        WHEN LAG(sales, 7) OVER p IS NULL
             OR LAG(sales, 7) OVER p = 0
        THEN NULL
        ELSE (sales - LAG(sales, 7) OVER p)
             / LAG(sales, 7) OVER p
    END AS percentage_change_7_days,

    CASE
        WHEN LAG(sales, 14) OVER p IS NULL
             OR LAG(sales, 14) OVER p = 0
        THEN NULL
        ELSE (sales - LAG(sales, 14) OVER p)
             / LAG(sales, 14) OVER p
    END AS percentage_change_14_days,

    CASE
        WHEN LAG(sales, 30) OVER p IS NULL
             OR LAG(sales, 30) OVER p = 0
        THEN NULL
        ELSE (sales - LAG(sales, 30) OVER p)
             / LAG(sales, 30) OVER p
    END AS percentage_change_30_days,

    CASE
        WHEN LAG(sales, 60) OVER p IS NULL
             OR LAG(sales, 60) OVER p = 0
        THEN NULL
        ELSE (sales - LAG(sales, 60) OVER p)
             / LAG(sales, 60) OVER p
    END AS percentage_change_60_days,

    CASE
        WHEN LAG(sales, 90) OVER p IS NULL
             OR LAG(sales, 90) OVER p = 0
        THEN NULL
        ELSE (sales - LAG(sales, 90) OVER p)
             / LAG(sales, 90) OVER p
    END AS percentage_change_90_days

FROM train

WINDOW
    p   AS (PARTITION BY store_nbr, family ORDER BY date),
    d_7  AS (PARTITION BY store_nbr, family ORDER BY date ROWS BETWEEN 6  PRECEDING AND CURRENT ROW),
    d_14 AS (PARTITION BY store_nbr, family ORDER BY date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW),
    d_30 AS (PARTITION BY store_nbr, family ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
    d_60 AS (PARTITION BY store_nbr, family ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
    d_90 AS (PARTITION BY store_nbr, family ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW)
""")

rows_6_and_7 = cur.fetchall()

cols_6_and_7 = [
    "date",
    "store_nbr",
    "family",
    "sales",
    "onpromotion",

    "mean_7_days",
    "mean_14_days",
    "mean_30_days",
    "mean_60_days",
    "mean_90_days",

    "std_7_days",
    "std_14_days",
    "std_30_days",
    "std_60_days",
    "std_90_days",

    "min_7_days",
    "max_7_days",
    "min_14_days",
    "max_14_days",
    "min_30_days",
    "max_30_days",
    "min_60_days",
    "max_60_days",
    "min_90_days",
    "max_90_days",

    "coefficient_var_7_days",
    "coefficient_var_14_days",
    "coefficient_var_30_days",
    "coefficient_var_60_days",
    "coefficient_var_90_days",

    "lag_1_day",
    "lag_2_days",
    "lag_3_days",
    "lag_7_days",
    "lag_14_days",
    "lag_28_days",
    "lag_365_days",

    "percentage_change_7_days",
    "percentage_change_14_days",
    "percentage_change_30_days",
    "percentage_change_60_days",
    "percentage_change_90_days",
]

df_6_and_7 = pd.DataFrame(rows_6_and_7, columns=cols_6_and_7)

df_6_and_7["date"] = pd.to_datetime(df_6_and_7["date"])
df_6_and_7 = df_6_and_7.sort_values(["store_nbr", "family", "date"])

g = df_6_and_7.groupby(["store_nbr", "family"], group_keys=False)

df_6_and_7["median_7_days"]  = g["sales"].rolling(7,  min_periods=1).median().reset_index(level=[0,1], drop=True)
df_6_and_7["median_14_days"] = g["sales"].rolling(14, min_periods=1).median().reset_index(level=[0,1], drop=True)
df_6_and_7["median_30_days"] = g["sales"].rolling(30, min_periods=1).median().reset_index(level=[0,1], drop=True)
df_6_and_7["median_60_days"] = g["sales"].rolling(60, min_periods=1).median().reset_index(level=[0,1], drop=True)
df_6_and_7["median_90_days"] = g["sales"].rolling(90, min_periods=1).median().reset_index(level=[0,1], drop=True)

df_6_and_7.to_csv("train_features.csv", index=False)

cur.close()
conn.close()


TASK 8

In [ ]:
# From transactions.csv, calculate rolling average transaction counts per store over 7 and 30 days.
# Join these to the main dataset and compute sales per transaction metrics.		
# Window AVG on transactions, join on store and date

# Читаем данные о транзакциях из CSV-файла
transaction = pd.read_csv('transactions.csv')

def rolling(transaction: pd.DataFrame):
    # Параметры подключения к базе данных
    DB_NAME = "postgres"
    DB_USER = "postgres"
    DB_PASS = pwd
    DB_HOST = "localhost"
    DB_PORT = "5432"
    
    # Устанавливаем соединение с PostgreSQL
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )
    # Устанавливаем режим автоподтверждения для немедленного выполнения команд
    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    cur = conn.cursor()
    
    # Удаляем существующую таблицу transaction, если она есть
    cur.execute("DROP TABLE IF EXISTS transaction;")
    
    # Создаем новую таблицу для данных о транзакциях
    # Используем составной первичный ключ (дата + номер магазина)
    cur.execute("""
        CREATE TABLE transaction (
            date DATE,
            store_nbr INT,
            transactions INT,
            PRIMARY KEY (date, store_nbr)
        );""")
    
    # Вставляем каждую строку из pandas DataFrame в SQL-таблицу
    # Используем плейсхолдеры %s для безопасной вставки (защита от SQL-инъекций)
    for _, row in transaction.iterrows():
        cur.execute(
            "INSERT INTO transaction VALUES (%s, %s, %s)",
            (row['date'], row['store_nbr'], row['transactions'])
        )
    
    # Подтверждаем вставку данных
    conn.commit()
    
    # Удаляем существующую таблицу со скользящими средними, если она есть
    cur.execute("DROP TABLE IF EXISTS transaction_with_rolling;")
    
    # Создаем новую таблицу с расчетом скользящих средних, используя оконные функции
    cur.execute("""
        CREATE TABLE transaction_with_rolling AS
        SELECT 
            date,
            store_nbr,
            transactions,
            -- Рассчитываем 7-дневное скользящее среднее: текущая строка + 6 предыдущих
            AVG(transactions) OVER (
                PARTITION BY store_nbr           -- Расчет отдельно для каждого магазина
                ORDER BY date                     -- Сортировка по дате для хронологического порядка
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW  -- Окно: последние 7 дней включая текущий
            ) as rolling_avg_7d,
            -- Рассчитываем 30-дневное скользящее среднее: текущая строка + 29 предыдущих
            AVG(transactions) OVER (
                PARTITION BY store_nbr
                ORDER BY date 
                ROWS BETWEEN 29 PRECEDING AND CURRENT ROW  -- Окно: последние 30 дней включая текущий
            ) as rolling_avg_30d
        FROM transaction;
    """)
    
    # Присоединяем исходные данные о транзакциях к рассчитанным скользящим средним
    cur.execute("""
        SELECT t.date, t.store_nbr, 
               r.rolling_avg_7d, r.rolling_avg_30d
        FROM transaction t
        INNER JOIN transaction_with_rolling r 
            ON t.date = r.date AND t.store_nbr = r.store_nbr  -- Присоединение по составному ключу
        ORDER BY t.store_nbr, t.date;  -- Сортировка по магазину и дате для удобства чтения
    """)
    
    # Получаем все результаты запроса
    data = cur.fetchall()
    
    # Преобразуем результаты в pandas DataFrame, обрабатывая пустой результат
    if not data:
        # Возвращаем пустой DataFrame с правильной структурой колонок, если данных нет
        result_df = pd.DataFrame(columns=['date', 'store_nbr', 'rolling_avg_7d', 'rolling_avg_30d'])
    else:
        # Извлекаем названия колонок из описания курсора и создаем DataFrame
        columns = [desc[0] for desc in cur.description]
        result_df = pd.DataFrame(data, columns=columns)
    
    # Очистка: удаляем временные таблицы, чтобы не засорять базу данных
    cur.execute("DROP TABLE transaction;")
    cur.execute("DROP TABLE transaction_with_rolling;")
    conn.commit()  # Подтверждаем удаление
    
    # Закрываем курсор и соединение с базой данных
    cur.close()
    conn.close()
    
    return result_df  # Возвращаем DataFrame со скользящими средними
    
rolling(transaction)

,date,store_nbr,rolling_avg_7d,rolling_avg_30d
0,2013-01-02,1,2111.0000000000000000,2111.0000000000000000
1,2013-01-03,1,1972.0000000000000000,1972.0000000000000000
2,2013-01-04,1,1935.6666666666666667,1935.6666666666666667
3,2013-01-05,1,1829.0000000000000000,1829.0000000000000000
4,2013-01-06,1,1567.2000000000000000,1567.2000000000000000
...,...,...,...,...
83483,2017-08-11,54,816.0000000000000000,792.7666666666666667
83484,2017-08-12,54,821.1428571428571429,800.1000000000000000
83485,2017-08-13,54,819.5714285714285714,812.7000000000000000
83486,2017-08-14,54,816.1428571428571429,811.4666666666666667


TASK 9

In [ ]:
# Seasonal Profile Creation: Calculate day-of-week average sales for each store-item over the entire history.
# Then calculate week-of-year average sales (seasonal profile).
# Create deviation features: current sales minus expected seasonal value.	
# Window functions with PARTITION BY store_nbr, item_nbr, day_of_week, AVG over all weeks		


import psycopg2
import pandas as pd
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from psycopg2.extras import execute_batch




def create_complete_seasonal_profiles(train_df: pd.DataFrame, test_df: pd.DataFrame = None,
                                      dbname='postgres', user='postgres', password='your_password',
                                      host='localhost', port='5432') -> pd.DataFrame:
    """
    Создаёт полные сезонные профили на основе тренировочных данных:
    - средние продажи по дням недели (avg_sales_by_dow)
    - средние продажи по неделям года (avg_sales_by_woy)
    - общее среднее (avg_sales_overall)
    - смешанное сезонное среднее (avg_sales_seasonal_blend)
    - различные отклонения от этих средних
    
    Возвращает обогащённый DataFrame с тренировочными данными.
    Если указан test_df, также создаёт признаки для тестовых данных.
    """
    # Устанавливаем соединение с базой данных PostgreSQL
    conn = psycopg2.connect(
        dbname=dbname,
        user=user,
        password=password,
        host=host,
        port=port
    )
    # Устанавливаем уровень изоляции для автоматического подтверждения транзакций
    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    # Создаем курсор для выполнения SQL-запросов
    cur = conn.cursor()
    
    # Удаляем временную таблицу train_data, если она существует (каскадно)
    cur.execute("DROP TABLE IF EXISTS train_data CASCADE;")
    # Удаляем временную таблицу test_data, если она существует (каскадно)
    cur.execute("DROP TABLE IF EXISTS test_data CASCADE;")
    
    # Создаем временную таблицу для train данных
    cur.execute("""
        CREATE TEMP TABLE train_data (
            id INT,                       -- идентификатор записи
            date DATE,                     -- дата
            store_nbr INT,                  -- номер магазина
            family VARCHAR(100),            -- категория товара
            sales DECIMAL,                   -- сумма продаж
            onpromotion INT                  -- количество товаров по акции
        );
    """)
    
    # Проходим по каждой строке train DataFrame
    for _, row in train_df.iterrows():
        # Вставляем данные в таблицу train_data
        cur.execute(
            "INSERT INTO train_data (id, date, store_nbr, family, sales, onpromotion) VALUES (%s, %s, %s, %s, %s, %s);",
            (int(row['id']), str(row['date']), int(row['store_nbr']), str(row['family']),
             float(row['sales']), int(row['onpromotion']))
        )
    
    # Если передан тестовый DataFrame
    if test_df is not None:
        # Создаем временную таблицу для тестовых данных
        cur.execute("""
            CREATE TEMP TABLE test_data (
                id INT,
                date DATE,
                store_nbr INT,
                family VARCHAR(100),
                onpromotion INT
            );
        """)
        # Проходим по каждой строке тестового DataFrame
        for _, row in test_df.iterrows():
            # Вставляем данные в таблицу test_data (без колонки sales)
            cur.execute(
                "INSERT INTO test_data (id, date, store_nbr, family, onpromotion) VALUES (%s, %s, %s, %s, %s);",
                (int(row['id']), str(row['date']), int(row['store_nbr']), str(row['family']), int(row['onpromotion'])))
    
    print("Выполнение SQL-запроса для создания полных сезонных профилей...")
    
    # SQL-запрос для создания сезонных профилей на train данных
    train_sql_query = """
    WITH base AS (
        -- Базовый слой: добавляем день недели и неделю года к каждой записи
        SELECT
            id,
            date,
            store_nbr,
            family,
            sales,
            onpromotion,
            EXTRACT(DOW FROM date) AS day_of_week,      -- день недели (0=воскресенье)
            EXTRACT(WEEK FROM date) AS week_of_year      -- неделя года по ISO
        FROM train_data
    ),
    dow_profile AS (
        -- Рассчитываем средние продажи по дням недели для каждого магазина и товара
        SELECT
            store_nbr,
            family,
            day_of_week,
            AVG(sales) AS avg_sales_by_dow
        FROM base
        GROUP BY store_nbr, family, day_of_week
    ),
    woy_profile AS (
        -- Рассчитываем средние продажи по неделям года для каждого магазина и товара
        SELECT
            store_nbr,
            family,
            week_of_year,
            AVG(sales) AS avg_sales_by_woy
        FROM base
        GROUP BY store_nbr, family, week_of_year
    ),
    overall_profile AS (
        -- Рассчитываем общее среднее продаж для каждого магазина и товара
        SELECT
            store_nbr,
            family,
            AVG(sales) AS avg_sales_overall
        FROM base
        GROUP BY store_nbr, family
    )
    -- Финальный SELECT: объединяем все данные и создаем признаки отклонения
    SELECT
        b.id,
        b.date,
        b.store_nbr,
        b.family,
        b.sales,
        b.onpromotion,
        -- Округляем средние значения до 2 знаков
        ROUND(d.avg_sales_by_dow::numeric, 2) AS avg_sales_by_dow,
        ROUND(w.avg_sales_by_woy::numeric, 2) AS avg_sales_by_woy,
        ROUND(o.avg_sales_overall::numeric, 2) AS avg_sales_overall,
        -- Отклонение от среднего по дню недели
        ROUND((b.sales - d.avg_sales_by_dow)::numeric, 2) AS sales_minus_dow_avg,
        -- Отклонение от среднего по неделе года
        ROUND((b.sales - w.avg_sales_by_woy)::numeric, 2) AS sales_minus_woy_avg,
        -- Отклонение от общего среднего
        ROUND((b.sales - o.avg_sales_overall)::numeric, 2) AS sales_minus_overall_avg,
        -- Смешанное сезонное среднее (комбинация дня недели и недели года)
        ROUND(((d.avg_sales_by_dow + w.avg_sales_by_woy) / 2.0)::numeric, 2) AS avg_sales_seasonal_blend,
        -- Отклонение от смешанного сезонного среднего
        ROUND((b.sales - ((d.avg_sales_by_dow + w.avg_sales_by_woy) / 2.0))::numeric, 2) AS sales_minus_seasonal_blend
    FROM base b
    -- LEFT JOIN с профилями по дням недели
    LEFT JOIN dow_profile d
        ON b.store_nbr = d.store_nbr
       AND b.family = d.family
       AND b.day_of_week = d.day_of_week
    -- LEFT JOIN с профилями по неделям года
    LEFT JOIN woy_profile w
        ON b.store_nbr = w.store_nbr
       AND b.family = w.family
       AND b.week_of_year = w.week_of_year
    -- LEFT JOIN с общими средними
    LEFT JOIN overall_profile o
        ON b.store_nbr = o.store_nbr
       AND b.family = o.family
    -- Сортировка по дате, магазину и категории товара
    ORDER BY b.date, b.store_nbr, b.family;
    """
    
    # Выполняем SQL-запрос для train данных
    cur.execute(train_sql_query)
    # Получаем все строки результата
    train_result = cur.fetchall()
    # Извлекаем названия колонок из описания курсора
    train_columns = [desc[0] for desc in cur.description]
    # Создаем pandas DataFrame из результатов
    train_result_df = pd.DataFrame(train_result, columns=train_columns)
    
    # Инициализируем переменную для тестовых результатов как None
    test_result_df = None
    
    # Если передан тестовый DataFrame
    if test_df is not None:
        # SQL-запрос для создания признаков на тестовых данных
        test_sql_query = """
        WITH base_train AS (
            -- Базовый слой из train данных для расчета профилей
            SELECT
                store_nbr,
                family,
                EXTRACT(DOW FROM date) AS day_of_week,
                EXTRACT(WEEK FROM date) AS week_of_year,
                sales
            FROM train_data
        ),
        dow_profile AS (
            -- Те же профили по дням недели (из тренировочных данных)
            SELECT
                store_nbr,
                family,
                day_of_week,
                AVG(sales) AS avg_sales_by_dow
            FROM base_train
            GROUP BY store_nbr, family, day_of_week
        ),
        woy_profile AS (
            -- Те же профили по неделям года (из тренировочных данных)
            SELECT
                store_nbr,
                family,
                week_of_year,
                AVG(sales) AS avg_sales_by_woy
            FROM base_train
            GROUP BY store_nbr, family, week_of_year
        ),
        overall_profile AS (
            -- Те же общие средние (из train данных)
            SELECT
                store_nbr,
                family,
                AVG(sales) AS avg_sales_overall
            FROM base_train
            GROUP BY store_nbr, family
        )
        -- Применяем профили к тестовым данным
        SELECT
            t.id,
            t.date,
            t.store_nbr,
            t.family,
            t.onpromotion,
            -- Добавляем средние значения из train данных
            ROUND(d.avg_sales_by_dow::numeric, 2) AS avg_sales_by_dow,
            ROUND(w.avg_sales_by_woy::numeric, 2) AS avg_sales_by_woy,
            ROUND(o.avg_sales_overall::numeric, 2) AS avg_sales_overall,
            -- Добавляем смешанное сезонное среднее
            ROUND(((d.avg_sales_by_dow + w.avg_sales_by_woy) / 2.0)::numeric, 2) AS avg_sales_seasonal_blend
        FROM test_data t
        -- LEFT JOIN для присоединения профилей по соответствующим ключам
        LEFT JOIN dow_profile d 
            ON t.store_nbr = d.store_nbr 
            AND t.family = d.family 
            AND EXTRACT(DOW FROM t.date) = d.day_of_week
        LEFT JOIN woy_profile w 
            ON t.store_nbr = w.store_nbr 
            AND t.family = w.family 
            AND EXTRACT(WEEK FROM t.date) = w.week_of_year
        LEFT JOIN overall_profile o
            ON t.store_nbr = o.store_nbr
            AND t.family = o.family
        ORDER BY t.id;
        """
        
        # Выполняем SQL-запрос для тестовых данных
        cur.execute(test_sql_query)
        # Получаем все строки результата
        test_result = cur.fetchall()
        # Извлекаем названия колонок
        test_columns = [desc[0] for desc in cur.description]
        # Создаем pandas DataFrame из результатов
        test_result_df = pd.DataFrame(test_result, columns=test_columns)
    
    # Удаляем временную таблицу train_data
    cur.execute("DROP TABLE IF EXISTS train_data CASCADE;")
    # Удаляем временную таблицу test_data
    cur.execute("DROP TABLE IF EXISTS test_data CASCADE;")

    # Закрываем курсор
    cur.close()
    # Закрываем соединение с базой данных
    conn.close()

    # Возвращаем результаты: если есть тестовые данные, возвращаем кортеж из двух DataFrame
    if test_result_df is not None:
        return train_result_df, test_result_df
    else:
        # Иначе возвращаем только train DataFrame
        return train_result_df


    
# Проверяем размеры
print(f"Train: {train.shape[0]} строк, {train.shape[1]} колонок")
print(f"Test: {test.shape[0]} строк, {test.shape[1]} колонок")
    
# Создаем признаки
train_features, test_features = create_complete_seasonal_profiles(
    train_df=train,
    test_df=test,
    password=pwd
    )
    
# Смотрим результат
print("\nTrain features (первые 5 строк):")
print(train_features.head())
print("\nTest features (первые 5 строк):")
print(test_features.head())
    
# Проверяем пропуски
print("\nПропуски в тестовых данных:")
for col in ['avg_sales_by_dow', 'avg_sales_by_woy', 'avg_sales_overall']:
    if col in test_features.columns:
        nulls = test_features[col].isna().sum()
        print(f"  {col}: {nulls} пропусков")

Train: 3000888 строк, 6 колонок
Test: 28512 строк, 5 колонок
Выполнение SQL-запроса для создания полных сезонных профилей...

Train features (первые 5 строк):
   id        date  store_nbr      family sales  onpromotion avg_sales_by_dow  \
0   0  2013-01-01          1  AUTOMOTIVE   0.0            0             3.87   
1   1  2013-01-01          1   BABY CARE   0.0            0             0.00   
2   2  2013-01-01          1      BEAUTY   0.0            0             2.59   
3   3  2013-01-01          1   BEVERAGES   0.0            0          1651.48   
4   4  2013-01-01          1       BOOKS   0.0            0             0.14   

  avg_sales_by_woy avg_sales_overall sales_minus_dow_avg sales_minus_woy_avg  \
0             2.91              3.25               -3.87               -2.91   
1             0.00              0.00                0.00                0.00   
2             2.41              2.41               -2.59               -2.41   
3          1631.03           1587.75    